# Enron EDA: corpus baseline, SPE terms, SPE subset, structural baseline, network and custodian coverage

Run all cells in order after `Enron_Data_Cleaning.ipynb` has produced a full run. Requires `pandas`, `matplotlib`, `networkx`, `pyarrow` (`pip install pandas matplotlib networkx pyarrow tzdata`). Outputs go to `eda_outputs/<run name>/`.

**Purpose.** Shape the search, not find the final emails. Each section ends with a decision the retrieval steps need.

| Section | Question | Decision it informs |
|---|---|---|
| 1. Corpus baseline | How big is the corpus, over which dates, in whose mailboxes, how long are the emails? | Denominators; whether to search new text or full text |
| 2. SPE terms and precision | Which proposal terms actually mean an SPE, and which are noise? | Which terms stand alone and which need financial context nearby |
| 3. SPE subset | How many distinct emails mention these terms? How many are quoted-only, external, before or after the cutoff? Are the seed emails found? | Size of the review workload; cutoff; text view |
| 5. Network and custodian coverage | Who sends, receives and holds the SPE emails? Which key people have mailboxes in the data? | Whose conduct the corpus can document; whom to prioritize |
| 7. Decision memo | What do sections 1 to 5 mean for the retrieval steps? | Written to `Enron_EDA_Decision_Memo.md`, with every number taken from this run |

**Not included here (by request):** monthly SPE rates with an event timeline, and contrastive term analysis. Section numbers keep the original plan so the group can add them later.

**Reading the results.**
- A *unique email* is a row with `is_duplicate_copy == False`. A *row* is one mailbox copy.
- Matching is on the full cleaned text unless stated; "new text" is `body_new_candidate`.
- Counts of term hits measure how often a word appears, not wrongdoing. Network position measures who handled these emails, not who was responsible.
- Dates and months use Houston local time.

## 0. Setup and data

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import json
import math
import random
import re

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import networkx as nx

try:
    from IPython.display import display
except ImportError:
    display = print
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 90)

# --- Paths ---
PROJECT_ROOT = Path.cwd()
OUTPUT_PARENT = PROJECT_ROOT / 'enron_cleaning_outputs'
INPUT_DIR = PROJECT_ROOT / 'eda_inputs'          # hand-made inputs (term review labels)
HOUSTON = ZoneInfo('America/Chicago')

# --- Analysis settings ---
CUTOFF_LOCAL = '2001-10-16T00:00:00'   # Houston time. Reference only: the group decides the real cutoff.
NEAR_CUTOFF_HOURS = 24                 # emails this close to the cutoff have uncertain dates (mailbox copies can be shifted)
PLOT_START, PLOT_END = '1998-01', '2002-12'
MASS_MAILING_THRESHOLD = 20            # emails to more than this many distinct addresses are left out of the network
CONTEXT_WINDOW_CHARS = 600             # about 100 words each side of a term (proposal: terms within one 200-word window)
SAMPLE_PER_STRATUM = 10                # emails reviewed per term and stratum
RANDOM_SEED = 6450
TOP_NODES = 25                         # people shown in network tables and charts

In [2]:
def find_latest_complete_run():
    """Newest run folder that has a report, a full-corpus flag, no parse errors and the 19-column schema (with body_retrieval)."""
    for folder in sorted((p for p in OUTPUT_PARENT.iterdir() if p.is_dir()), reverse=True):
        report_file, table_file = folder / 'quality_report.json', folder / 'emails_analysis.parquet'
        if not (report_file.exists() and table_file.exists()):
            continue   # interrupted run
        report = json.loads(report_file.read_text(encoding='utf-8'))
        if (report.get('is_full_corpus') and not report.get('parse_errors')
                and 'body_retrieval' in pq.ParquetFile(table_file).schema_arrow.names):
            return folder, report
    raise FileNotFoundError('No complete full-corpus run found; run Enron_Data_Cleaning.ipynb first.')


RUN_DIR, run_report = find_latest_complete_run()
EDA_DIR = PROJECT_ROOT / 'eda_outputs' / RUN_DIR.name
EDA_DIR.mkdir(parents=True, exist_ok=True)
TABLE = RUN_DIR / 'emails_analysis.parquet'

META_COLUMNS = ['email_id', 'canonical_email_id', 'is_duplicate_copy', 'custodian', 'folder', 'date_utc', 'sender',
                'recipients', 'cc', 'bcc', 'subject', 'subject_norm', 'quote_marker_type', 'quality_flags']
meta = pq.read_table(TABLE, columns=META_COLUMNS).to_pandas()
text_table = pq.read_table(TABLE, columns=['body_clean', 'body_new_candidate', 'body_retrieval'])
body_full, body_new, body_retr = (text_table.column(c) for c in ['body_clean', 'body_new_candidate', 'body_retrieval'])
assert len(meta) == len(body_full) == run_report['counts']['valid_records']

# --- Derived columns used throughout ---
meta['date_local'] = meta['date_utc'].dt.tz_convert(HOUSTON)
meta['month'] = meta['date_local'].dt.tz_localize(None).dt.to_period('M')
meta['recipient_count'] = [len(set(a) | set(b) | set(c)) for a, b, c in zip(meta['recipients'], meta['cc'], meta['bcc'])]
meta['internal_sender'] = [len(s) > 0 and all('enron' in a.rsplit('@', 1)[-1] for a in s) for s in meta['sender']]
meta['custodian_key'] = meta['custodian'].str.casefold()     # SKILLING-J and Skilling-J are the same mailbox
unique_mask = ~meta['is_duplicate_copy'].to_numpy()          # one row per distinct email
email_ids = meta['email_id'].to_numpy()


def cutoff_utc(houston_local):
    """Houston local time without an offset -> UTC (daylight saving applied: midnight Oct 16, 2001 CDT = 05:00 UTC)."""
    return datetime.fromisoformat(houston_local).replace(tzinfo=HOUSTON).astimezone(timezone.utc)


CUTOFF_UTC = pd.Timestamp(cutoff_utc(CUTOFF_LOCAL))
assert CUTOFF_UTC == pd.Timestamp('2001-10-16 05:00:00', tz='UTC')
facts = {'run': RUN_DIR.name}          # numbers collected here feed the decision memo


def save_table(frame, name, index=False):
    frame.to_csv(EDA_DIR / name, index=index)


def save_figure(figure, name):
    figure.savefig(EDA_DIR / name, dpi=130, bbox_inches='tight')


print(f'Run {RUN_DIR.name}: {len(meta):,} rows; cutoff {CUTOFF_LOCAL} Houston = {CUTOFF_UTC}')

Run 20261113T083630222199Z: 251,734 rows; cutoff 2001-10-16T00:00:00 Houston = 2001-10-16 05:00:00+00:00


## 1. Corpus baseline
Denominators for everything else: how many distinct emails, how many mailbox copies, which dates, whose mailboxes, how long the texts are.

In [3]:
unique_meta = meta[unique_mask]
dated = unique_meta['date_utc'].notna()
outside = lambda lo, hi: int((dated & ((unique_meta['month'] < pd.Period(lo)) | (unique_meta['month'] > pd.Period(hi)))).sum())

facts.update({
    'rows': len(meta), 'unique_emails': int(unique_mask.sum()), 'duplicate_copies': int((~unique_mask).sum()),
    'custodians': int(meta['custodian_key'].nunique()), 'custodian_labels': int(meta['custodian'].nunique()),
    'no_date_unique': int((~dated).sum()),
    'outside_plot_range': outside(PLOT_START, PLOT_END),
    'distinct_senders': len({a for s in unique_meta['sender'] for a in s}),
})
totals = pd.DataFrame([
    ('Rows in analysis table (mailbox copies)', facts['rows']),
    ('Unique emails (is_duplicate_copy == False)', facts['unique_emails']),
    ('Duplicate copies', facts['duplicate_copies']),
    ('Share of rows that are duplicate copies', f"{facts['duplicate_copies'] / facts['rows']:.1%}"),
    ('Custodians (mailboxes, labels case-folded)', facts['custodians']),
    ('Custodian labels before case-folding', facts['custodian_labels']),
    ('Distinct sender addresses (unique emails)', facts['distinct_senders']),
    ('Unique emails with no date', facts['no_date_unique']),
    ('Earliest date (Houston)', str(unique_meta['date_local'].min())[:10]),
    ('Latest date (Houston)', str(unique_meta['date_local'].max())[:10]),
    (f'Unique emails outside {PLOT_START} to {PLOT_END}', facts['outside_plot_range']),
], columns=['measure', 'value'])
display(totals)
save_table(totals, 'baseline_totals.csv')

                                       measure       value
0      Rows in analysis table (mailbox copies)      251734
1   Unique emails (is_duplicate_copy == False)      249243
2                             Duplicate copies        2491
3      Share of rows that are duplicate copies        1.0%
4   Custodians (mailboxes, labels case-folded)         158
5         Custodian labels before case-folding         258
6    Distinct sender addresses (unique emails)       20326
7                   Unique emails with no date           0
8                      Earliest date (Houston)  1979-12-31
9                        Latest date (Houston)  2044-01-04
10    Unique emails outside 1998-01 to 2002-12         458


In [4]:
span = pd.period_range(PLOT_START, PLOT_END, freq='M')
monthly_emails = unique_meta.groupby('month').size().reindex(span, fill_value=0)
monthly_custodians = meta.groupby('month')['custodian_key'].nunique().reindex(span, fill_value=0)
monthly = pd.DataFrame({'unique_emails': monthly_emails, 'active_custodians': monthly_custodians})
save_table(monthly, 'baseline_monthly.csv', index=True)

figure, axis = plt.subplots(figsize=(11, 4.5))
axis.bar(span.to_timestamp(), monthly_emails.values, width=25, color='#4c72b0', label='Unique emails')
axis.set_ylabel('Unique emails per month')
second = axis.twinx()
second.plot(span.to_timestamp(), monthly_custodians.values, color='#dd8452', lw=2, label='Active custodians')
second.set_ylabel('Mailboxes with email that month')
axis.axvline(CUTOFF_UTC.tz_convert(HOUSTON).tz_localize(None), color='black', ls='--', lw=1)
axis.text(CUTOFF_UTC.tz_convert(HOUSTON).tz_localize(None), axis.get_ylim()[1] * 0.95, ' cutoff', ha='left', va='top')
axis.set_title('Corpus volume and mailbox coverage by month (Houston time)')
figure.legend(loc='upper left', bbox_to_anchor=(0.08, 0.88))
save_figure(figure, 'baseline_monthly.png')
plt.show()

peak = monthly_emails.idxmax()
facts.update({'peak_month': str(peak), 'peak_month_emails': int(monthly_emails.max()),
              'peak_month_custodians': int(monthly_custodians[peak])})
print(f"Peak month {peak}: {facts['peak_month_emails']:,} unique emails from {facts['peak_month_custodians']} mailboxes")

<baseline-monthly>:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Peak month 2001-10: 27,628 unique emails from 137 mailboxes


In [5]:
custodian_rows = meta['custodian_key'].fillna('(missing)').value_counts()
facts.update({'custodian_rows_median': int(custodian_rows.median()),
              'top10_custodian_share': float(custodian_rows.head(10).sum() / len(meta)),
              'missing_custodian_rows': int(meta['custodian'].isna().sum())})
print(f"{len(custodian_rows)} custodians (labels case-folded); median {facts['custodian_rows_median']:,} rows per mailbox; "
      f"top 10 hold {facts['top10_custodian_share']:.1%} of rows; {facts['missing_custodian_rows']} rows without a custodian")
save_table(custodian_rows.rename('rows').rename_axis('custodian').reset_index(), 'baseline_custodians.csv')

figure, axis = plt.subplots(figsize=(7, 5))
custodian_rows.head(15)[::-1].plot.barh(ax=axis, color='#4c72b0')
axis.set_xlabel('Rows (mailbox copies)')
axis.set_title('Largest mailboxes')
save_figure(figure, 'baseline_custodians.png')
plt.show()

159 custodians (labels case-folded); median 938 rows per mailbox; top 10 hold 32.0% of rows; 2 rows without a custodian
<baseline-custodians>:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [6]:
full_len = np.asarray(pc.utf8_length(body_full).to_numpy(), dtype=np.int64)
new_len = np.asarray(pc.utf8_length(body_new).to_numpy(), dtype=np.int64)
marker = meta['quote_marker_type'].fillna('none')
u = unique_mask

lengths = pd.DataFrame({
    'full text (characters)': pd.Series(full_len[u]).quantile([.5, .75, .9, .95, .99, 1]).round(0),
    'new text (characters)': pd.Series(new_len[u]).quantile([.5, .75, .9, .95, .99, 1]).round(0),
}).rename(index={0.5: 'median', 0.75: '75th pct', 0.9: '90th pct', 0.95: '95th pct', 0.99: '99th pct', 1.0: 'max'})
display(lengths)

with_marker = u & (marker.to_numpy() != 'none')
ratio = new_len[with_marker] / np.maximum(full_len[with_marker], 1)
facts.update({
    'share_with_quote_marker': float(with_marker.sum() / u.sum()),
    'share_new_text_empty': float((new_len[u] == 0).sum() / u.sum()),
    'share_quote_heavy': float((ratio < 0.5).mean()),       # among emails with a marker: under half is new text
    'median_full_chars': int(np.median(full_len[u])), 'p99_full_chars': int(np.quantile(full_len[u], .99)),
    'share_mass_mailing': float((meta.loc[u, 'recipient_count'] > MASS_MAILING_THRESHOLD).mean()),
    'share_external_sender': float((~meta.loc[u, 'internal_sender']).mean()),
})
print(f"Emails with a quote marker: {facts['share_with_quote_marker']:.1%}; new text empty: {facts['share_new_text_empty']:.1%}")
print(f"Among emails with a marker, {facts['share_quote_heavy']:.1%} are under half new text")
print(f"Mass mailings (> {MASS_MAILING_THRESHOLD} recipients): {facts['share_mass_mailing']:.1%}; external-only senders: {facts['share_external_sender']:.1%}")

marker_table = marker[u].value_counts().rename('unique_emails').rename_axis('quote_marker_type').reset_index()
flag_table = pd.Series(Counter(f for flags in meta.loc[u, 'quality_flags'] for f in flags)).sort_values(ascending=False)
flag_table = flag_table.rename('unique_emails').rename_axis('quality_flag').reset_index()
display(marker_table)
display(flag_table)
save_table(lengths, 'baseline_lengths.csv', index=True)
save_table(marker_table, 'baseline_quote_markers.csv')
save_table(flag_table, 'baseline_quality_flags.csv')

          full text (characters)  new text (characters)
median                     753.0                  200.0
75th pct                  1719.0                  571.0
90th pct                  3574.0                 1577.0
95th pct                  5781.0                 3297.0
99th pct                 15605.0                13189.0
max                    2011420.0              2011420.0
Emails with a quote marker: 43.7%; new text empty: 7.5%
Among emails with a marker, 95.7% are under half new text
Mass mailings (> 20 recipients): 4.7%; external-only senders: 21.4%
          quote_marker_type  unique_emails
0                      none         140366
1  outlook_original_message          53261
2              forwarded_by          29735
3        notes_header_block          25704
4      outlook_header_block             89
5                  on_wrote             69
6         forwarded_message             19
                    quality_flag  unique_emails
0           attachment_mentioned  

## 2. SPE terms and precision
The proposal's Table 2 Group A names, plus a few candidate additions that appear in the Powers report. For each term this section measures how often it appears, how often only in quoted text, and how often it appears near a financial-structure word (Table 2 Groups B and C).

**Precision** here means: of the emails containing the word, what share use it in the Enron SPE or structured-finance sense (not a person, place, product or other company). It is not a measure of wrongdoing. A stratified sample is reviewed for each term: emails with a financial context word within `CONTEXT_WINDOW_CHARS` of the term, and emails without. Comparing the two shows whether requiring context helps.

Labels are in `eda_inputs/term_review_labels.csv`: `Y` the excerpt refers to the Enron vehicle or structured/related-party transaction of that name, `N` another meaning (person, place, product, other company, or for Cuiabá the plant project itself rather than the LJM transaction), `U` cannot tell. Excerpts are about 200 characters, so some labels are judgment calls. They were assigned by one reviewer and should be spot-checked by the group. Ten emails per stratum give wide intervals: read the interval, not just the point estimate.

In [7]:
# label, entity group, regular expression (word boundaries, case-insensitive), kind, source
TERM_ROWS = [
    ('raptor',           'Raptor vehicles', r'\braptors?\b',                                         'proposal', 'Proposal Table 2; Powers (2002). Also covers Raptor I-IV and 1-4.'),
    ('talon',            'Raptor vehicles', r'\btalon\b',                                            'proposal', 'Proposal Table 2; Powers (2002)'),
    ('porcupine',        'Raptor vehicles', r'\bporcupine\b',                                        'proposal', 'Proposal Table 2; Powers (2002)'),
    ('ljm',              'LJM',             r'\bljm\d?\b',                                           'proposal', 'Proposal Table 2; Powers (2002). Also covers LJM1, LJM2, LJM Cayman, LJM Co-Investment.'),
    ('chewco',           'Chewco',          r'\bchewco\b',                                           'proposal', 'Proposal Table 2; Powers (2002)'),
    ('jedi',             'JEDI',            r'\bjedi\b',                                             'proposal', 'Proposal Table 2; Powers (2002). Also covers JEDI II.'),
    ('hawaii_125',       'Hawaii 125-0',    r'\bhawaii[ -]?(?:sec(?:tion|\.)?[ -]?)?125(?:-0)?\b',   'proposal', 'Proposal Table 2 (Hawaii 125-0, Hawaii Sec 125); Pruett (2001)'),
    ('hawaii',           'Hawaii (any)',    r'\bhawaii\b',                                           'proposal', 'Proposal Table 2; also a place name'),
    ('whitewing',        'Whitewing',       r'\bwhitewing\b',                                        'proposal', 'Proposal Table 2; Powers (2002)'),
    ('condor',           'Condor',          r'\bcondor\b',                                           'proposal', 'Proposal Table 2; Powers (2002)'),
    ('osprey',           'Osprey',          r'\bosprey\b',                                           'proposal', 'Proposal Table 2; Powers (2002)'),
    ('marlin',           'Marlin',          r'\bmarlin\b',                                           'proposal', 'Proposal Table 2; not found in Powers (2002)'),
    ('project_velocity', 'Project Velocity', r'\bproject velocity\b',                               'proposal', 'Proposal Table 2; not found in Powers (2002)'),
    ('trutta',           'Trutta',          r'\btrutta\b',                                           'proposal', 'Proposal Table 2; not found in Powers (2002)'),
    ('rhythms',          'Rhythms',         r'\brhythms\b',                                          'proposal', 'Proposal Table 2; Powers (2002) (Rhythms NetConnections)'),
    ('timberwolf',       'Raptor vehicles', r'\btimber[ -]?wolf\b',                                  'candidate', 'Candidate addition: named in Powers (2002); needs page citation'),
    ('bobcat',           'Raptor vehicles', r'\bbobcat\b',                                           'candidate', 'Candidate addition: named in Powers (2002); needs page citation'),
    ('southampton',      'Southampton',     r'\bsouthampton\b',                                      'candidate', 'Candidate addition: named in Powers (2002); needs page citation'),
    ('cuiaba',           'Cuiaba',          r'\bcuiab[aá]',                                          'candidate', 'Candidate addition: named in Powers (2002); needs page citation'),
]
TERMS = pd.DataFrame(TERM_ROWS, columns=['label', 'entity', 'pattern', 'kind', 'source'])
ENTITY = dict(zip(TERMS['label'], TERMS['entity']))
TERM_RE = {row.label: re.compile(row.pattern, re.IGNORECASE) for row in TERMS.itertuples()}

# Financial-structure words from proposal Table 2 Groups B (retained exposure) and C (accounting).
# Used only to check whether a term sits in a financial context.
CONTEXT_TERMS = [
    r'guarant(?:y|ee|ees|ied|eed|ying|eeing)', r'recourse', r'indemni\w+', r'make[ -]?whole', r'repurchas\w+', r'buy[ -]?back',
    r'total[ -]return[ -]swap', r'contingent obligation', r'credit (?:support|capacity|reserve)', r'collateral\w*',
    r'(?:enron|ene) stock', r'equity at risk', r'side (?:agreement|letter)', r'off[ -]balance[ -]sheet', r'(?:de)?consolidat\w+',
    r'hedg\w+', r'mark[ -]to[ -]market', r'mtm', r'valuation', r'write[ -]?down', r'impairment', r'loss recognition',
    r'disclosure', r'footnote', r'auditor', r'independent third party',
]
CONTEXT_RE = re.compile('|'.join(f'(?:{p})' for p in CONTEXT_TERMS), re.IGNORECASE)
print(len(TERMS), 'terms;', len(CONTEXT_TERMS), 'context patterns')

19 terms; 26 context patterns


In [8]:
import time

def regex_hits(column, pattern):
    """True for each row whose text matches (RE2 engine, case-insensitive). Fast and vectorised."""
    return np.asarray(pc.match_substring_regex(column, pattern, ignore_case=True).to_numpy(), dtype=bool)


started = time.time()
hits_full = {row.label: regex_hits(body_full, row.pattern) for row in TERMS.itertuples()}
hits_new = {row.label: regex_hits(body_new, row.pattern) for row in TERMS.itertuples()}
print(f'Matched {len(TERMS)} terms on both text views in {time.time() - started:.0f}s')


def get_texts(column, indices):
    return column.take(pa.array(np.asarray(indices, dtype=np.int64))).to_pylist()


def occurrence_info(text, term_re, window=CONTEXT_WINDOW_CHARS):
    """(first occurrence, first occurrence with a context word within +-window characters); each (start, end) or None."""
    first = first_with_context = None
    for match in term_re.finditer(text):
        if first is None:
            first = (match.start(), match.end())
        if CONTEXT_RE.search(text, max(0, match.start() - window), min(len(text), match.end() + window)):
            first_with_context = (match.start(), match.end())
            break
    return first, first_with_context


# For every unique email containing the term: does a financial context word sit near it?
unique_hit_idx, has_context = {}, {}
for label in TERMS['label']:
    idx = np.flatnonzero(hits_full[label] & unique_mask)
    unique_hit_idx[label] = idx
    infos = [occurrence_info(t, TERM_RE[label]) for t in get_texts(body_full, idx)] if len(idx) else []
    has_context[label] = np.array([ctx is not None for _, ctx in infos], dtype=bool)
print('Context check done for', sum(len(v) for v in unique_hit_idx.values()), 'email-term hits')

Matched 19 terms on both text views in 34s
Context check done for 2643 email-term hits


In [9]:
rows = []
for row in TERMS.itertuples():
    f, n = hits_full[row.label], hits_new[row.label]
    unique_hits = int((f & unique_mask).sum())
    rows.append({
        'term': row.label, 'entity': row.entity, 'kind': row.kind,
        'rows_with_term': int(f.sum()), 'unique_emails': unique_hits,
        'in_new_text': int((n & unique_mask).sum()),
        'quoted_only_share': (unique_hits - int((n & unique_mask).sum())) / unique_hits if unique_hits else np.nan,
        'with_context_share': has_context[row.label].mean() if unique_hits else np.nan,
    })
term_stats = pd.DataFrame(rows)
display(term_stats.style.format({'quoted_only_share': '{:.0%}', 'with_context_share': '{:.0%}'}) if hasattr(term_stats, 'style') else term_stats)
save_table(term_stats, 'term_stats.csv')

                term            entity       kind  rows_with_term  unique_emails  in_new_text  quoted_only_share  with_context_share
0             raptor   Raptor vehicles   proposal             353            344          173           0.497093            0.470930
1              talon   Raptor vehicles   proposal              48             48           25           0.479167            0.416667
2          porcupine   Raptor vehicles   proposal              16             16            3           0.812500            0.375000
3                ljm               LJM   proposal             326            315          167           0.469841            0.498413
4             chewco            Chewco   proposal              84             75           56           0.253333            0.586667
5               jedi              JEDI   proposal             167            166          110           0.337349            0.409639
6         hawaii_125      Hawaii 125-0   proposal              52    

In [10]:
def snippet(text, span, width=110):
    start, end = span
    piece = f"{text[max(0, start - width):start]}[[{text[start:end]}]]{text[end:end + width]}"
    return re.sub(r'\s+', ' ', piece).strip()


review_rows = []
for row in TERMS.itertuples():
    idx, ctx = unique_hit_idx[row.label], has_context[row.label]
    for stratum, flag in [('context', True), ('no_context', False)]:
        pool = sorted(idx[ctx == flag].tolist(), key=lambda i: email_ids[i])        # stable order, independent of row order
        chosen = random.Random(f'{RANDOM_SEED}-{row.label}-{stratum}').sample(pool, min(SAMPLE_PER_STRATUM, len(pool)))
        for i, text in zip(chosen, get_texts(body_full, chosen) if chosen else []):
            first, first_ctx = occurrence_info(text, TERM_RE[row.label])
            span = first_ctx if (flag and first_ctx) else first
            review_rows.append({'term': row.label, 'stratum': stratum, 'email_id': email_ids[i],
                                'date': str(meta.at[i, 'date_local'])[:10], 'subject': meta.at[i, 'subject'][:70],
                                'excerpt': snippet(text, span)})
review = pd.DataFrame(review_rows)

LABELS_PATH = INPUT_DIR / 'term_review_labels.csv'
if LABELS_PATH.exists():
    labels = pd.read_csv(LABELS_PATH, dtype=str)[['term', 'stratum', 'email_id', 'label']]
    review = review.merge(labels, on=['term', 'stratum', 'email_id'], how='left')
else:
    review['label'] = np.nan
save_table(review, 'term_review_sample.csv')
print(f'{len(review)} sampled emails; labelled: {int(review["label"].notna().sum())}; file: {EDA_DIR / "term_review_sample.csv"}')

340 sampled emails; labelled: 340; file: D:\E_Guo\z_New_folder\INTA6450\enron_proj\eda_outputs\20261113T083630222199Z\term_review_sample.csv


In [11]:
def wilson(k, n, z=1.96):
    """95% Wilson interval for a proportion k/n."""
    if n == 0:
        return (np.nan, np.nan)
    p, denom = k / n, 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))


precision_rows = []
for row in TERMS.itertuples():
    population = {'context': int(has_context[row.label].sum()), 'no_context': int((~has_context[row.label]).sum())}
    record = {'term': row.label, 'unique_emails': sum(population.values()), 'with_context': population['context'], 'without_context': population['no_context']}
    estimates = {}
    for stratum in ['context', 'no_context']:
        sample = review[(review['term'] == row.label) & (review['stratum'] == stratum)]
        yes, no = int((sample['label'] == 'Y').sum()), int((sample['label'] == 'N').sum())
        low, high = wilson(yes, yes + no)
        record[f'{stratum}_Y/(Y+N)'] = f'{yes}/{yes + no}' if yes + no else ''
        record[f'{stratum}_precision'] = yes / (yes + no) if yes + no else np.nan
        record[f'{stratum}_ci'] = f'{low:.2f}-{high:.2f}' if yes + no else ''
        estimates[stratum] = record[f'{stratum}_precision']
    # Overall precision: strata weighted by how many emails each contains in the corpus
    parts = [(population[s], estimates[s]) for s in population if population[s] and not np.isnan(estimates[s])]
    covered = sum(n for n, _ in parts)
    record['overall_precision_est'] = sum(n * p for n, p in parts) / covered if covered and covered == record['unique_emails'] else np.nan
    precision_rows.append(record)
precision = pd.DataFrame(precision_rows)
display(precision)
save_table(precision, 'term_precision.csv')

                term  unique_emails  with_context  without_context context_Y/(Y+N)  context_precision context_ci no_context_Y/(Y+N)  no_context_precision no_context_ci  overall_precision_est
0             raptor            344           162              182           10/10           1.000000  0.72-1.00               9/10                   0.9     0.60-0.98               0.947093
1              talon             48            20               28            9/10           0.900000  0.60-0.98               5/10                   0.5     0.24-0.76               0.666667
2          porcupine             16             6               10             6/6           1.000000  0.61-1.00               5/10                   0.5     0.24-0.76               0.687500
3                ljm            315           157              158           10/10           1.000000  0.72-1.00              10/10                   1.0     0.72-1.00               1.000000
4             chewco             75          

## 3. SPE subset
Unique emails that mention an SPE term, using the term classes below. **Strong** terms count on their own. **Context-required** terms count only when a financial-structure word (proposal Table 2 Groups B or C) is within `CONTEXT_WINDOW_CHARS`. **Dropped** terms are ignored because their SPE sense is too rare in this corpus.

The classes come from the precision table in Section 2 (`eda_outputs/<run>/term_precision.csv`). Change the lists if the group's labels differ.

In [12]:
STRONG_TERMS = ['raptor', 'ljm', 'chewco', 'jedi', 'hawaii_125', 'whitewing', 'condor', 'trutta', 'project_velocity']
CONTEXT_REQUIRED_TERMS = ['talon', 'porcupine', 'osprey', 'marlin', 'rhythms', 'timberwolf', 'bobcat', 'southampton']
DROPPED_TERMS = ['hawaii', 'cuiaba']

assert set(STRONG_TERMS) | set(CONTEXT_REQUIRED_TERMS) | set(DROPPED_TERMS) == set(TERMS['label']), 'every term needs a class'
assert not (set(STRONG_TERMS) & set(CONTEXT_REQUIRED_TERMS)) and not ({*STRONG_TERMS, *CONTEXT_REQUIRED_TERMS} & set(DROPPED_TERMS))
print('strong:', STRONG_TERMS)
print('context required:', CONTEXT_REQUIRED_TERMS)
print('dropped:', DROPPED_TERMS)

strong: ['raptor', 'ljm', 'chewco', 'jedi', 'hawaii_125', 'whitewing', 'condor', 'trutta', 'project_velocity']
context required: ['talon', 'porcupine', 'osprey', 'marlin', 'rhythms', 'timberwolf', 'bobcat', 'southampton']
dropped: ['hawaii', 'cuiaba']


In [13]:
# --- Which unique emails belong to the subset, and which terms matched (full text) ---
terms_full = defaultdict(list)
for label in STRONG_TERMS:
    for i in unique_hit_idx[label]:
        terms_full[int(i)].append(label)
for label in CONTEXT_REQUIRED_TERMS:
    for i, ok in zip(unique_hit_idx[label], has_context[label]):
        if ok:
            terms_full[int(i)].append(label)
subset_idx = np.array(sorted(terms_full), dtype=np.int64)

# --- Same rule on the new text only (does the sender's own text contain the term?) ---
terms_new = {int(i): [] for i in subset_idx}
for label in STRONG_TERMS:
    for i in unique_hit_idx[label]:
        if hits_new[label][i]:
            terms_new[int(i)].append(label)
for label in CONTEXT_REQUIRED_TERMS:
    candidates = [int(i) for i in subset_idx if label in terms_full[int(i)] and hits_new[label][i]]
    for i, text in zip(candidates, get_texts(body_new, candidates) if candidates else []):
        if occurrence_info(text, TERM_RE[label])[1] is not None:
            terms_new[i].append(label)

spe = meta.loc[subset_idx, ['email_id', 'canonical_email_id', 'custodian', 'custodian_key', 'date_utc', 'sender', 'recipients', 'cc', 'bcc',
                            'subject', 'subject_norm', 'internal_sender', 'recipient_count']].copy()
spe['terms'] = [terms_full[int(i)] for i in subset_idx]
spe['terms_new'] = [terms_new[int(i)] for i in subset_idx]
spe['entities'] = spe['terms'].map(lambda ts: sorted({ENTITY[t] for t in ts}))
spe['in_new_text'] = spe['terms_new'].map(bool)
spe['mailbox_copies'] = spe['canonical_email_id'].map(meta.groupby('canonical_email_id').size())

date = spe['date_utc']
spe['period'] = np.select([date.isna(), date < CUTOFF_UTC], ['no date', 'before cutoff'], default='on or after cutoff')
spe['near_cutoff'] = ((date - CUTOFF_UTC).abs() <= pd.Timedelta(hours=NEAR_CUTOFF_HOURS)).to_numpy()
assert len(spe) == len(set(spe['canonical_email_id'])), 'subset must hold each email once'
print(f'{len(spe):,} unique emails in the SPE subset')

911 unique emails in the SPE subset


In [14]:
u_date = unique_meta['date_utc']
corpus_period = pd.Series(np.select([u_date.isna(), u_date < CUTOFF_UTC], ['no date', 'before cutoff'], default='on or after cutoff')).value_counts()
spe_period = spe['period'].value_counts()
after_share_spe = spe_period.get('on or after cutoff', 0) / len(spe)
after_share_corpus = corpus_period.get('on or after cutoff', 0) / len(unique_meta)

facts.update({
    'spe_unique': len(spe), 'spe_share_of_corpus': len(spe) / facts['unique_emails'],
    'spe_with_copies': int((spe['mailbox_copies'] > 1).sum()),
    'spe_in_new_text': int(spe['in_new_text'].sum()), 'spe_quoted_only': int((~spe['in_new_text']).sum()),
    'spe_external': int((~spe['internal_sender']).sum()), 'spe_mass': int((spe['recipient_count'] > MASS_MAILING_THRESHOLD).sum()),
    'spe_threads': int(spe['subject_norm'].nunique()), 'spe_senders': len({a for s in spe['sender'] for a in s}),
    'spe_custodians': int(spe['custodian_key'].nunique()),
    'spe_before_cutoff': int(spe_period.get('before cutoff', 0)), 'spe_after_cutoff': int(spe_period.get('on or after cutoff', 0)),
    'spe_no_date': int(spe_period.get('no date', 0)), 'spe_near_cutoff': int(spe['near_cutoff'].sum()),
    'spe_after_share': float(after_share_spe), 'corpus_after_share': float(after_share_corpus),
    'multi_entity': int((spe['entities'].map(len) > 1).sum()),
})
profile = pd.DataFrame([
    ('Unique emails in subset', facts['spe_unique']),
    ('Share of all unique emails', f"{facts['spe_share_of_corpus']:.2%}"),
    ('Distinct subjects after removing RE/FW (thread proxy)', facts['spe_threads']),
    ('Distinct sender addresses', facts['spe_senders']),
    ('Custodians holding a first copy', facts['spe_custodians']),
    ('Emails also held in other mailboxes (copies)', facts['spe_with_copies']),
    ('Term found in new text', facts['spe_in_new_text']),
    ('Quoted-only (term appears only in quoted or forwarded text)', facts['spe_quoted_only']),
    ('Sender outside Enron', facts['spe_external']),
    (f'Mass mailings (> {MASS_MAILING_THRESHOLD} recipients)', facts['spe_mass']),
    ('Matches more than one entity group', facts['multi_entity']),
    (f'Before cutoff ({CUTOFF_LOCAL} Houston)', facts['spe_before_cutoff']),
    ('On or after cutoff', facts['spe_after_cutoff']),
    ('No date', facts['spe_no_date']),
    (f'Within {NEAR_CUTOFF_HOURS} hours of the cutoff (date uncertain)', facts['spe_near_cutoff']),
    ('Share of subset on or after cutoff', f"{after_share_spe:.1%}"),
    ('Share of whole corpus on or after cutoff', f"{after_share_corpus:.1%}"),
], columns=['measure', 'value'])
display(profile)
save_table(profile, 'spe_profile.csv')

# Who is left for review? Sender inside Enron, term in the sender's own text, not a mass mailing; split by the cutoff
spe['prime'] = spe['internal_sender'] & spe['in_new_text'] & (spe['recipient_count'] <= MASS_MAILING_THRESHOLD)
period_table = spe.groupby('period').agg(emails=('email_id', 'size'), internal_sender=('internal_sender', 'sum'),
                                         term_in_new_text=('in_new_text', 'sum'), prime_review_pool=('prime', 'sum')).reset_index()
display(period_table)
save_table(period_table, 'spe_by_period.csv')
facts['spe_prime_total'] = int(spe['prime'].sum())
facts['spe_prime_before'] = int(spe.loc[spe['period'] == 'before cutoff', 'prime'].sum())

entity_rows = []
for label in STRONG_TERMS + CONTEXT_REQUIRED_TERMS:
    in_subset = spe['terms'].map(lambda ts: label in ts)
    entity_rows.append({'term': label, 'class': 'strong' if label in STRONG_TERMS else 'context required',
                        'emails': int(in_subset.sum()), 'in_new_text': int(spe.loc[in_subset, 'terms_new'].map(lambda ts: label in ts).sum())})
entity_table = pd.DataFrame(entity_rows).sort_values('emails', ascending=False)
display(entity_table)
save_table(entity_table, 'spe_terms_in_subset.csv')

# Cost of the class rules: emails that mention a term but were left out, and roughly how many of them are real mentions
in_subset_rows = set(int(i) for i in spe.index)
precision_by_term = precision.set_index('term')


def labelled_interval(term, strata):
    """95% interval for the share of real mentions, from the labelled emails of the given strata."""
    sample = review[(review['term'] == term) & review['stratum'].isin(strata)]
    return wilson(int((sample['label'] == 'Y').sum()), int(sample['label'].isin(['Y', 'N']).sum()))


gate_rows = []
for label in CONTEXT_REQUIRED_TERMS + DROPPED_TERMS:
    gated = label in CONTEXT_REQUIRED_TERMS
    excluded = unique_hit_idx[label][~has_context[label]] if gated else unique_hit_idx[label]
    share_real = precision_by_term.at[label, 'no_context_precision'] if gated else precision_by_term.at[label, 'overall_precision_est']
    low, high = labelled_interval(label, ['no_context'] if gated else ['context', 'no_context'])
    only_here = [i for i in excluded if int(i) not in in_subset_rows]       # not rescued by another term
    gate_rows.append({'term': label, 'class': 'context required' if gated else 'dropped',
                      'emails_left_out': len(excluded), 'left_out_and_not_in_subset': len(only_here),
                      'share_real_mentions_in_sample': share_real,
                      'est_real_mentions_lost': round(len(only_here) * share_real) if not np.isnan(share_real) else np.nan,
                      'approx_range': f'{round(len(only_here) * low)}-{round(len(only_here) * high)}' if not np.isnan(low) else ''})
gate_cost = pd.DataFrame(gate_rows)
display(gate_cost)
save_table(gate_cost, 'term_class_cost.csv')
facts['gate_cost'] = gate_cost[['term', 'left_out_and_not_in_subset', 'est_real_mentions_lost']].to_dict('records')
facts['est_real_mentions_lost'] = float(gate_cost['est_real_mentions_lost'].sum())
save_table(spe.drop(columns=['terms_new']).assign(terms=spe['terms'].map(', '.join), entities=spe['entities'].map(', '.join),
                                                  sender=spe['sender'].map(', '.join), recipients=spe['recipients'].map(', '.join),
                                                  cc=spe['cc'].map(', '.join), bcc=spe['bcc'].map(', '.join)), 'spe_subset_emails.csv')

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
entity_table.set_index('term')['emails'][::-1].plot.barh(ax=axes[0], color='#4c72b0')
axes[0].set_title('SPE subset: unique emails per term'); axes[0].set_xlabel('Emails')
split = pd.Series({'in new text': facts['spe_in_new_text'], 'quoted only': facts['spe_quoted_only']})
split.plot.bar(ax=axes[1], color=['#55a868', '#c44e52'], rot=0)
axes[1].set_title('Where the term appears')
save_figure(figure, 'spe_subset_overview.png')
plt.show()

                                                        measure  value
0                                       Unique emails in subset    911
1                                    Share of all unique emails  0.37%
2         Distinct subjects after removing RE/FW (thread proxy)    472
3                                     Distinct sender addresses    320
4                               Custodians holding a first copy     88
5                  Emails also held in other mailboxes (copies)     12
6                                        Term found in new text    471
7   Quoted-only (term appears only in quoted or forwarded text)    440
8                                          Sender outside Enron     67
9                               Mass mailings (> 20 recipients)     38
10                           Matches more than one entity group    241
11                  Before cutoff (2001-10-16T00:00:00 Houston)    529
12                                           On or after cutoff    382
13    

In [15]:
# Known emails from the proposal. They shaped the dictionary, so finding them is a sanity check, not an accuracy test.
SEEDS = [
    ('Kaminski: "LJM/Raptor valuations" (Oct 4, 2001)', 'LJM/Raptor valuations', '2001-10-04'),
    ('Pruett: "CGAS Hawaii Sec 125 Trust" (May 2, 2001)', 'CGAS Hawaii Sec 125 Trust', '2001-05-02'),
]
spe_ids = set(spe['canonical_email_id'])
seed_rows = []
for name, subject, day in SEEDS:
    target = pd.Timestamp(day, tz=HOUSTON)
    found = meta[meta['subject_norm'].str.contains(re.escape(subject.casefold()), regex=True)
                 & ((meta['date_local'] - target).abs() <= pd.Timedelta(days=1.5))]
    if found.empty:
        seed_rows.append({'seed': name, 'found': False})
        continue
    for i, row in found.iterrows():
        ungated = [l for l in TERMS['label'] if hits_full[l][i]]
        seed_rows.append({'seed': name, 'found': True, 'email_id': row['email_id'], 'copy': bool(row['is_duplicate_copy']),
                          'custodian': row['custodian'], 'date_local': str(row['date_local'])[:16], 'sender': ', '.join(row['sender']),
                          'recipients': ', '.join(row['recipients'][:3]), 'in_subset': row['canonical_email_id'] in spe_ids,
                          'terms_in_text': ', '.join(ungated),
                          'subset_terms': ', '.join(spe.loc[i, 'terms']) if i in spe.index else ''})
seed_table = pd.DataFrame(seed_rows)
display(seed_table)
save_table(seed_table, 'seed_check.csv')
facts['seeds_found'] = int(seed_table.get('found', pd.Series(dtype=bool)).sum()) if len(seed_table) else 0
facts['seeds_total'] = len(SEEDS)
facts['seeds_in_subset'] = int(seed_table['in_subset'].eq(True).groupby(seed_table['seed']).any().sum()) if 'in_subset' in seed_table else 0

                                                seed  found                          email_id   copy   custodian        date_local                sender               recipients  in_subset       terms_in_text  \
0    Kaminski: "LJM/Raptor valuations" (Oct 4, 2001)   True  c9732ecec3d1333cbf8ce9d461b7fe7f  False       Buy-R  2001-10-04 08:27  j.kaminski@enron.com    ryan.siurek@enron.com       True         raptor, ljm   
1  Pruett: "CGAS Hawaii Sec 125 Trust" (May 2, 2001)   True  f6a682d86455b20afb09bfde1adf2b76  False  Lavorado-J  2001-05-02 16:40      pruett@enron.com  john.lavorato@enron.com       True  hawaii_125, hawaii   

  subset_terms  
0  raptor, ljm  
1   hawaii_125  


## 4. Structural baseline (search set for TF-IDF and semantic search)

The baseline keeps emails that look like readable internal Enron correspondence. **It uses metadata and format only, with no SPE terms, people or relevance scores**, so it does not decide in advance what wrongdoing looks like. TF-IDF is fitted on, and semantic search runs over, `body_retrieval` for these emails.

| Rule | Keep an email when | Why |
|---|---|---|
| B1 Unique | `is_duplicate_copy == False` | Rank each email once; copies stay reachable through `canonical_email_id`. |
| B2 Internal sender | every From address is at an Enron domain | Focus on what employees wrote; drops newsletters, vendors and spam. |
| B3 Not a broadcast | ? `MASS_MAILING_THRESHOLD` (20) distinct To/CC/BCC addresses, and the sender is not on `BROADCAST_SENDERS` | Company-wide notices distort term weights. |
| B4 Has retrieval text | `body_retrieval` has non-whitespace text | Both channels score this field. Emails that fail B4 alone go to a **fallback set**, searched on `body_clean` (e.g. forwards without comment). |
| B5 Dated, in study window | `date_utc` present and Houston date in [`BASELINE_START`, `BASELINE_END`) | The period when the SPEs were active; the Oct 16, 2001 cutoff is **not** applied here. |

Excluded emails are not deleted. The recall check below runs the Section 3 SPE rules on them, to show what the baseline costs.

In [16]:
# --- Settings: proposed defaults for the group to confirm ---
BASELINE_START, BASELINE_END = '1999-01-01', '2002-01-01'     # Houston local dates, end exclusive
BROADCAST_SENDERS = {'enron.announcements@enron.com', 'announcements.enron@enron.com', 'enron.general.announcements@enron.com',
                     'chairman.office@enron.com', 'office.chairman@enron.com', 'no.address@enron.com', 'all.worldwide@enron.com'}

start, end = pd.Timestamp(BASELINE_START, tz=HOUSTON), pd.Timestamp(BASELINE_END, tz=HOUSTON)
has_retrieval = np.asarray(pc.utf8_length(pc.utf8_trim_whitespace(body_retr)).to_numpy(zero_copy_only=False) > 0)
rules = {
    'B1 unique': unique_mask,
    'B2 internal sender': meta['internal_sender'].to_numpy(),
    'B3 not a broadcast': ((meta['recipient_count'] <= MASS_MAILING_THRESHOLD)
                           & ~meta['sender'].map(lambda s: bool(set(s) & BROADCAST_SENDERS))).to_numpy(),
    'B4 has retrieval text': has_retrieval,
    'B5 dated, in window': (meta['date_local'].notna() & (meta['date_local'] >= start) & (meta['date_local'] < end)).to_numpy(),
}

# --- Funnel: rules applied in order, plus what each rule removes on its own (from unique emails) ---
keep, funnel_rows = np.ones(len(meta), bool), [('all records', len(meta), '')]
for name, mask in rules.items():
    keep &= mask
    alone = '' if name.startswith('B1') else int((unique_mask & ~mask).sum())
    funnel_rows.append((name, int(keep.sum()), alone))
baseline_funnel = pd.DataFrame(funnel_rows, columns=['after rule', 'emails left', 'unique emails failing this rule alone'])
display(baseline_funnel)
save_table(baseline_funnel, 'baseline_funnel.csv')
baseline_mask = keep
fallback_mask = rules['B1 unique'] & rules['B2 internal sender'] & rules['B3 not a broadcast'] & rules['B5 dated, in window'] & ~has_retrieval

# Broadcast list check: entries that never occur are harmless but should be pruned
sender_counts = Counter(a for s in meta.loc[unique_mask, 'sender'] for a in s)
print('BROADCAST_SENDERS found:', {a: sender_counts[a] for a in sorted(BROADCAST_SENDERS) if sender_counts[a]})
heavy = meta[unique_mask & (meta['recipient_count'] > MASS_MAILING_THRESHOLD).to_numpy()]
print('Top senders of mail to >20 addresses (candidates for the list):',
      Counter(a for s in heavy['sender'] for a in s).most_common(8))

# --- Saved baseline: email ids plus a 'set' column (primary or fallback) ---
baseline_ids = pd.concat([pd.DataFrame({'email_id': email_ids[baseline_mask], 'set': 'primary'}),
                          pd.DataFrame({'email_id': email_ids[fallback_mask], 'set': 'fallback'})], ignore_index=True)
baseline_ids.to_parquet(EDA_DIR / 'structural_baseline.parquet', index=False)
retr_len = np.asarray(pc.utf8_length(body_retr).to_numpy(zero_copy_only=False))
new_len = np.asarray(pc.utf8_length(body_new).to_numpy(zero_copy_only=False))
print(f"Primary baseline {baseline_mask.sum():,} emails; fallback {fallback_mask.sum():,}. "
      f"body_retrieval is shorter than body_new_candidate in {(retr_len < new_len)[baseline_mask].mean():.1%} of primary emails; "
      f"median length {int(np.median(retr_len[baseline_mask])):,} characters.")

              after rule  emails left unique emails failing this rule alone
0            all records       251734                                      
1              B1 unique       249243                                      
2     B2 internal sender       195781                                 53462
3     B3 not a broadcast       185142                                 13474
4  B4 has retrieval text       168411                                 18657
5    B5 dated, in window       152885                                 27667
BROADCAST_SENDERS found: {'announcements.enron@enron.com': 102, 'chairman.office@enron.com': 14, 'enron.announcements@enron.com': 756, 'no.address@enron.com': 712, 'office.chairman@enron.com': 68}
Top senders of mail to >20 addresses (candidates for the list): [('jeff.dasovich@enron.com', 287), ('lorna.brennan@enron.com', 184), ('ginger.dernehl@enron.com', 178), ('susan.mara@enron.com', 129), ('miyung.buster@enron.com', 117), ('sgovenar@govadv.com', 116), ('cheryl

In [17]:
# --- Recall check: which SPE-subset emails (Section 3) does the baseline exclude, and why? ---
order = list(rules)
def first_failed(i):
    if baseline_mask[i]:
        return 'in primary baseline'
    if fallback_mask[i]:
        return 'fallback (no retrieval text)'
    return next(name for name in order if not rules[name][i])
spe_status = spe.assign(baseline=[first_failed(i) for i in spe.index])
spe_recall = spe_status['baseline'].value_counts().rename_axis('status').reset_index(name='SPE emails')
display(spe_recall)
save_table(spe_recall, 'baseline_spe_recall.csv')

# Seeds: is each known email in the baseline? (canonical copy of any matching row)
row_of = pd.Series(np.arange(len(meta)), index=meta['email_id'])
seed_status = []
for r in seed_table[seed_table.get('found', False) == True].itertuples():
    canonical = meta.at[row_of[r.email_id], 'canonical_email_id']
    seed_status.append({'seed': r.seed, 'email_id': r.email_id, 'baseline': first_failed(int(row_of[canonical]))})
seed_status = pd.DataFrame(seed_status).drop_duplicates(['seed', 'baseline'])
display(seed_status)

# Sample of excluded SPE emails for hand review (do these matter?)
excluded = spe_status[~spe_status['baseline'].isin(['in primary baseline', 'fallback (no retrieval text)'])]
review = excluded.sample(frac=1, random_state=RANDOM_SEED).groupby('baseline').head(5).sort_values('baseline')
review = review[['email_id', 'baseline', 'date_utc', 'sender', 'recipient_count', 'subject', 'terms']]
save_table(review, 'baseline_excluded_spe_sample.csv')
display(review)

kept_spe = int((spe_status['baseline'] == 'in primary baseline').sum())
facts.update({'baseline_primary': int(baseline_mask.sum()), 'baseline_fallback': int(fallback_mask.sum()),
              'baseline_spe_kept': kept_spe, 'baseline_spe_total': len(spe)})
def baseline_memo():   # called by the memo cell, where md_table is defined
    return f"""
## Structural baseline (Section 4)

Search set for TF-IDF and semantic search: unique emails (B1) from an Enron sender (B2), to at most {MASS_MAILING_THRESHOLD} addresses and not from a broadcast account (B3), with non-empty `body_retrieval` (B4), dated in [{BASELINE_START}, {BASELINE_END}) Houston time (B5). No topic terms are used.

{md_table(baseline_funnel)}

- Primary baseline: **{baseline_mask.sum():,}** emails. Fallback set (passes all rules but B4; search on `body_clean`): **{fallback_mask.sum():,}**.
- Recall check: **{kept_spe:,} of {len(spe):,}** SPE-subset emails are in the primary baseline. Where the others go:

{md_table(spe_recall)}

- Excluded SPE emails are sampled in `baseline_excluded_spe_sample.csv` for the group to read. If they matter (e.g. Andersen or Vinson & Elkins writing in), relax B2 or B3, or search them separately.
"""

                         status  SPE emails
0           in primary baseline         623
1  fallback (no retrieval text)         121
2            B2 internal sender          67
3           B5 dated, in window          58
4            B3 not a broadcast          37
5         B4 has retrieval text           5
                                                seed                          email_id             baseline
0    Kaminski: "LJM/Raptor valuations" (Oct 4, 2001)  c9732ecec3d1333cbf8ce9d461b7fe7f  in primary baseline
1  Pruett: "CGAS Hawaii Sec 125 Trust" (May 2, 2001)  f6a682d86455b20afb09bfde1adf2b76  in primary baseline
                                email_id               baseline                  date_utc                              sender  recipient_count  \
211406  d70517f88209086e0ec4d31e8892bf14     B2 internal sender 2001-11-15 16:50:46+00:00  [richard.a.sibery@us.andersen.com]                3   
75706   4d22aaefcad22b53abf6ee9b735f8d0f     B2 internal sender 2001-10-25 0

## 5. Network and custodian coverage
Who sent, received and held the SPE emails.

**How to read it**
- The corpus holds the mailboxes of about 150 people. Anyone else appears only through other people's mail, so a person's position reflects whose mailboxes were collected as much as who handled the deals.
- Central does not mean culpable. People who reviewed or objected to a transaction are central too.
- The network uses unique SPE emails, leaves out mass mailings and self-only messages, and does not merge address aliases (except the two Kaminski aliases listed in the key-people cell).
- Which addresses belong to custodians is inferred, not recorded. **Sent-folder** evidence is strong (the address is the only sender of mail in that custodian's Sent folder). **Name match** (first.last@enron.com against the label `Last-F`) is weaker, because two people can share a surname and initial. Results that depend on it are labelled.
- Mailbox copies are better evidence of receipt than the recipient fields: a mailbox can hold an email its owner was never addressed on.

In [18]:
# Which addresses belong to custodians? An address that is the only sender of mail in a custodian's Sent folder.
sent_folder = meta['folder'].fillna('').str.contains(r'(?i)(?<![a-z])sent(?![a-z])', regex=True)
single_sender = meta['sender'].map(len) == 1
own_sent = sent_folder & single_sender & meta['custodian_key'].notna()
pairs = pd.DataFrame({'address': meta.loc[own_sent, 'sender'].map(lambda s: s[0]), 'custodian': meta.loc[own_sent, 'custodian_key']})
pair_counts = pairs.value_counts().rename('n').reset_index()
by_address = pair_counts.groupby('address').agg(total=('n', 'sum'), top=('n', 'max'))
by_address['custodian'] = pair_counts.sort_values('n', ascending=False).drop_duplicates('address').set_index('address')['custodian']
by_address['share'] = by_address['top'] / by_address['total']
custodian_addresses = by_address[(by_address['total'] >= 3) & (by_address['share'] >= 0.7)]
addr_to_custodian = custodian_addresses['custodian'].to_dict()


def custodian_name_parts(label):
    """'Kaminski-V' -> ('kaminski', 'v'); 'WILLIAMS-W3' -> ('williams', 'w')."""
    last, _, rest = label.rpartition('-')
    return (last.lower(), re.sub(r'\d', '', rest)[:1].lower()) if last else (label.lower(), '')


agree = np.mean([custodian_name_parts(c)[0] in a.split('@')[0].lower() for a, c in addr_to_custodian.items()])
facts.update({'mapped_addresses': len(addr_to_custodian), 'mapped_custodians': len(set(addr_to_custodian.values())),
              'name_agreement': float(agree)})
print(f"Sent-folder rule: {len(addr_to_custodian):,} addresses mapped to {facts['mapped_custodians']} of {facts['custodians']} custodians; "
      f"custodian surname appears in the address for {agree:.1%} of mappings (validation of the rule)")

# Fallback for custodians with no usable Sent folder: match 'first.last@enron.com' or 'flast@enron.com' to the custodian label.
# Weaker evidence (two people can share a surname and initial), so it is reported separately.
custodian_by_name = {}
for key in meta['custodian_key'].dropna().unique():
    custodian_by_name.setdefault(custodian_name_parts(key), key)
name_cache = {}


def custodian_of(address, strict=False):
    """(custodian key, evidence) for an address: 'sent folder', 'name match', or (None, None)."""
    if address in addr_to_custodian:
        return addr_to_custodian[address], 'sent folder'
    if strict:
        return None, None
    if address not in name_cache:
        local, _, domain = address.partition('@')
        tokens = [t for t in re.split(r'[._\-]+', local.lower()) if t]
        guesses = []
        if 'enron' in domain and len(tokens) >= 2:
            guesses.append((tokens[-1].rstrip('0123456789'), tokens[0][0]))
        if 'enron' in domain and len(tokens) == 1 and len(tokens[0]) > 3:
            guesses.append((tokens[0][1:], tokens[0][0]))
        name_cache[address] = next(((custodian_by_name[g], 'name match') for g in guesses if g in custodian_by_name), (None, None))
    return name_cache[address]


facts['custodians_with_sent_address'] = len(set(addr_to_custodian.values()))
print(f"{len(custodian_by_name)} custodian name keys available for the name-match fallback")

Sent-folder rule: 205 addresses mapped to 143 of 158 custodians; custodian surname appears in the address for 83.4% of mappings (validation of the rule)
158 custodian name keys available for the name-match fallback


In [19]:
# People named in the Powers report or the proposal's example emails (first initial, surname)
KEY_PEOPLE = [
    ('Andrew Fastow', 'a', 'fastow', 'Powers'), ('Richard Causey', 'r', 'causey', 'Powers'), ('Ben Glisan', 'b', 'glisan', 'Powers'),
    ('Michael Kopper', 'm', 'kopper', 'Powers'), ('Jeffrey McMahon', 'j', 'mcmahon', 'Powers'), ('Jeffrey Skilling', 'j', 'skilling', 'Powers'),
    ('Kenneth Lay', 'k', 'lay', 'Powers'), ('Richard Buy', 'r', 'buy', 'Powers'), ('James Derrick', 'j', 'derrick', 'Powers'),
    ('Jordan Mintz', 'j', 'mintz', 'Powers'), ('Vince Kaminski', 'v', 'kaminski', 'Powers; proposal example'),
    ('Sherron Watkins', 's', 'watkins', 'Powers'), ('Ryan Siurek', 'r', 'siurek', 'Proposal example'),
    ('John Lavorato', 'j', 'lavorato', 'Proposal example'), ('Steve Pruett', 's', 'pruett', 'Proposal example'),
]


def address_matches(address, initial, last):
    """Enron address whose local part is first.last, last, or initial+last (e.g. vkaminski). No other aliases."""
    local, _, domain = address.partition('@')
    if 'enron' not in domain or last not in local.lower():
        return False
    tokens = [t for t in re.split(r'[._\-]+', local.lower()) if t]
    if not tokens:
        return False
    if re.fullmatch(rf'{initial}{last}\d*', tokens[-1]):
        return True
    return tokens[-1].rstrip('0123456789') == last and (len(tokens) == 1 or tokens[0].startswith(initial))


def email_counters(frame):
    """Distinct emails per address as sender, and as recipient (To, CC or BCC)."""
    sent, received = Counter(), Counter()
    for s, r, c, b in zip(frame['sender'], frame['recipients'], frame['cc'], frame['bcc']):
        sent.update(set(s))
        received.update(set(r) | set(c) | set(b))
    return sent, received


corpus_sent, corpus_received = email_counters(unique_meta)
spe_sent, spe_received = email_counters(spe)
all_addresses = set(corpus_sent) | set(corpus_received)
cust_rows = meta['custodian_key'].value_counts()
spe_canonical = set(spe['canonical_email_id'])
copy_rows = meta[meta['canonical_email_id'].isin(spe_canonical)]

# Aliases seen in the data that the name rule cannot match (checked against the Oct 4, 2001 seed email and the Raptor excerpts)
EXTRA_ALIASES = {'Vince Kaminski': ['j.kaminski@enron.com', 'vkaminski@aol.com']}

people_rows, person_addresses = [], {}
for name, initial, last, basis in KEY_PEOPLE:
    addresses = [a for a in all_addresses if address_matches(a, initial, last)]
    addresses += [a for a in EXTRA_ALIASES.get(name, []) if a in all_addresses and a not in addresses]
    person_addresses[name] = addresses
    custodians = [c for c in cust_rows.index if custodian_name_parts(c) == (last, initial)]
    held = copy_rows[copy_rows['custodian_key'].isin(custodians)]['canonical_email_id'].nunique() if custodians else 0
    sent_n, recv_n = sum(corpus_sent[a] for a in addresses), sum(corpus_received[a] for a in addresses)
    spe_sent_n, spe_recv_n = sum(spe_sent[a] for a in addresses), sum(spe_received[a] for a in addresses)
    status = 'custodian' if custodians else ('appears in addresses only' if (sent_n + recv_n) else 'absent')
    people_rows.append({
        'person': name, 'basis': basis, 'status': status, 'custodian_label': ', '.join(custodians),
        'mailbox_rows': int(sum(cust_rows[c] for c in custodians)),
        'addresses_matched': ', '.join(sorted(addresses, key=lambda a: -(corpus_sent[a] + corpus_received[a]))[:3]),
        'emails_sent_corpus': sent_n, 'emails_received_corpus': recv_n,
        'spe_sent': spe_sent_n, 'spe_received': spe_recv_n, 'spe_emails_held_in_mailbox': held,
    })
key_people = pd.DataFrame(people_rows)
display(key_people)
save_table(key_people, 'key_people_coverage.csv')
facts['key_people'] = key_people[['person', 'status', 'spe_sent', 'spe_received', 'spe_emails_held_in_mailbox']].to_dict('records')
facts['key_status_counts'] = key_people['status'].value_counts().to_dict()

              person                     basis                     status custodian_label  mailbox_rows                                                             addresses_matched  emails_sent_corpus  \
0      Andrew Fastow                    Powers  appears in addresses only                             0     andrew.fastow@enron.com, andy.fastow@enron.com, andrew.s.fastow@enron.com                   8   
1     Richard Causey                    Powers  appears in addresses only                             0            richard.causey@enron.com, rcausey@enron.com, rick.causey@enron.com                  26   
2         Ben Glisan                    Powers  appears in addresses only                             0               ben.glisan@enron.com, bglisan@enron.com, ben_f_glisan@enron.com                   8   
3     Michael Kopper                    Powers  appears in addresses only                             0                                                      michael.kopper@enron.co

In [20]:
# --- Clean the subset for the network: one copy of each email, no mass mailings, no self-only messages ---
funnel = [('SPE subset (unique emails)', len(spe))]
usable = spe[(spe['sender'].map(len) > 0) & (spe['recipient_count'] > 0)]
funnel.append(('with a sender and at least one recipient', len(usable)))
usable = usable[usable['recipient_count'] <= MASS_MAILING_THRESHOLD]
funnel.append((f'not a mass mailing (<= {MASS_MAILING_THRESHOLD} recipients)', len(usable)))

edge_weight, sent_by, received_by = Counter(), Counter(), Counter()
handled = defaultdict(set)          # address -> emails it sent or received
kept_emails = 0
for row in usable.itertuples():
    senders, recipients = set(row.sender), set(row.recipients) | set(row.cc) | set(row.bcc)
    pairs_here = [(s, r) for s in senders for r in recipients if s != r]
    if not pairs_here:
        continue                      # only the sender addressed themselves
    kept_emails += 1
    for s, r in pairs_here:
        edge_weight[(s, r)] += 1
    for s in senders:
        sent_by[s] += 1; handled[s].add(row.email_id)
    for r in recipients - senders:
        received_by[r] += 1; handled[r].add(row.email_id)
funnel.append(('with at least one sender-to-recipient link', kept_emails))
funnel_table = pd.DataFrame(funnel, columns=['step', 'emails'])
display(funnel_table)
save_table(funnel_table, 'network_funnel.csv')

G = nx.DiGraph()
G.add_weighted_edges_from((s, r, w) for (s, r), w in edge_weight.items())
U = G.to_undirected()
betweenness = nx.betweenness_centrality(U, k=min(len(U), 400), seed=RANDOM_SEED)


def node_type(address):
    key, evidence = custodian_of(address)
    if key:
        return 'custodian' if evidence == 'sent folder' else 'custodian (name match)'
    return 'Enron address, no mailbox' if 'enron' in address.split('@')[-1] else 'outside Enron'


nodes = pd.DataFrame({
    'address': list(G.nodes),
    'type': [node_type(a) for a in G.nodes],
    'custodian': [custodian_of(a)[0] or '' for a in G.nodes],
    'emails_handled': [len(handled[a]) for a in G.nodes],
    'emails_sent': [sent_by[a] for a in G.nodes],
    'emails_received': [received_by[a] for a in G.nodes],
    'correspondents': [U.degree(a) for a in G.nodes],
    'betweenness': [betweenness[a] for a in G.nodes],
}).sort_values('emails_handled', ascending=False).reset_index(drop=True)
nodes['rank'] = np.arange(1, len(nodes) + 1)
save_table(nodes, 'network_nodes.csv')
print(f'{G.number_of_nodes():,} addresses, {G.number_of_edges():,} links; betweenness is exact' if len(U) <= 400 else
      f'{G.number_of_nodes():,} addresses, {G.number_of_edges():,} links; betweenness estimated from 400 sampled sources')
display(nodes.head(TOP_NODES))

type_share = nodes.head(TOP_NODES)['type'].value_counts()
facts.update({'network_nodes': G.number_of_nodes(), 'network_edges': G.number_of_edges(), 'network_emails': kept_emails,
              'top_type_counts': type_share.to_dict(), 'top_nodes': nodes.head(10)[['address', 'type', 'emails_handled']].to_dict('records')})

                                         step  emails
0                  SPE subset (unique emails)     911
1    with a sender and at least one recipient     829
2       not a mass mailing (<= 20 recipients)     791
3  with at least one sender-to-recipient link     752
779 addresses, 1,387 links; betweenness estimated from 400 sampled sources
                         address                       type     custodian  emails_handled  emails_sent  emails_received  correspondents  betweenness  rank
0      sara.shackleton@enron.com                  custodian  shackleton-s             210           73              137              58     0.114036     1
1            mary.cook@enron.com  Enron address, no mailbox                            99           46               53              63     0.135512     2
2          mark.taylor@enron.com                  custodian      taylor-m              41            8               33              19     0.009739     3
3          carol.clair@enron.com   

In [21]:
# Mailbox holders that were not on the email: evidence of receipt that the address fields do not show.
# Strict: an address counts as a custodian's only with Sent-folder evidence. Extended: also name matches.
# Strict overstates the gap (some custodians have no mapped address); extended may understate it (shared surname and initial).
holders = copy_rows.groupby('canonical_email_id')['custodian_key'].agg(lambda s: set(s.dropna()))


def unaddressed_holders(strict):
    result = {}
    for row in spe.itertuples():
        addressed = {custodian_of(a, strict)[0] for a in set(row.sender) | set(row.recipients) | set(row.cc) | set(row.bcc)}
        extra = holders.get(row.canonical_email_id, set()) - addressed
        if extra:
            result[row.canonical_email_id] = extra
    return result


unaddressed_strict, unaddressed = unaddressed_holders(True), unaddressed_holders(False)
extra_counter = Counter(c for s in unaddressed.values() for c in s)
facts.update({'spe_unaddressed_strict': len(unaddressed_strict), 'spe_unaddressed_holder_emails': len(unaddressed),
              'spe_unaddressed_share': len(unaddressed) / len(spe), 'spe_unaddressed_strict_share': len(unaddressed_strict) / len(spe),
              'top_unaddressed_holders': extra_counter.most_common(8)})
print(f"{len(unaddressed):,} of {len(spe):,} SPE emails ({facts['spe_unaddressed_share']:.1%}) sit in a mailbox whose owner is not "
      f"the sender or an addressee under the extended address map ({facts['spe_unaddressed_strict_share']:.1%} under the strict map)")
print('Most frequent such mailboxes (extended map):', extra_counter.most_common(8))

copies_table = spe['mailbox_copies'].clip(upper=5).value_counts().sort_index().rename('emails').rename_axis('mailbox_copies').reset_index()
display(copies_table)
save_table(copies_table, 'spe_mailbox_copies.csv')
save_table(pd.DataFrame([{'canonical_email_id': k, 'unaddressed_mailboxes': ', '.join(sorted(v))} for k, v in unaddressed.items()]), 'spe_unaddressed_holders.csv')

171 of 911 SPE emails (18.8%) sit in a mailbox whose owner is not the sender or an addressee under the extended address map (20.5% under the strict map)
Most frequent such mailboxes (extended map): [('martin-t', 37), ('kitchen-l', 35), ('whalley-g', 15), ('bass-e', 9), ('whalley-l', 6), ('dean-c', 6), ('salisbury-h', 5), ('swerzbin-m', 5)]
   mailbox_copies  emails
0               1     899
1               2      12


In [22]:
palette = {'custodian': '#4c72b0', 'custodian (name match)': '#8fb0d6', 'Enron address, no mailbox': '#dd8452', 'outside Enron': '#8c8c8c'}
key_addresses = {a for addrs in person_addresses.values() for a in addrs}
top = nodes.head(TOP_NODES)

figure, axes = plt.subplots(1, 2, figsize=(15, 6.5), gridspec_kw={'width_ratios': [1, 1.25]})
axes[0].barh(top['address'].str.split('@').str[0][::-1], top['emails_handled'][::-1], color=[palette[t] for t in top['type'][::-1]])
axes[0].set_xlabel('SPE emails handled (sent or received)')
axes[0].set_title(f'Top {TOP_NODES} addresses by SPE emails handled')
axes[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in palette.values()], labels=list(palette), loc='lower right', fontsize=8)

sub = U.subgraph(top['address']).copy()
positions = nx.spring_layout(sub, seed=RANDOM_SEED, weight='weight', k=0.9)
for kind, colour in palette.items():
    members = [a for a in sub.nodes if node_type(a) == kind]
    nx.draw_networkx_nodes(sub, positions, nodelist=members, node_color=colour, ax=axes[1],
                           node_size=[60 + 12 * len(handled[a]) ** 0.7 for a in members],
                           edgecolors=['red' if a in key_addresses else 'white' for a in members], linewidths=2)
nx.draw_networkx_edges(sub, positions, ax=axes[1], alpha=0.25, width=0.6)
nx.draw_networkx_labels(sub, positions, labels={a: a.split('@')[0] for a in sub.nodes}, font_size=6.5, ax=axes[1])
axes[1].set_title('Links among the top addresses (red outline: person named in the Powers report or proposal)')
axes[1].axis('off')
save_figure(figure, 'network_overview.png')
plt.show()

<network-plots>:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 7. Decision memo
Turns the results above into recommendations for the retrieval steps. Numbers are filled in from this run; the wording is fixed, so re-read it if the data or labels change. It is a snapshot for the group to discuss, not a final decision.

In [23]:
def md_table(frame):
    """Plain markdown table (no extra packages)."""
    lines = ['| ' + ' | '.join(frame.columns) + ' |', '|' + '|'.join(['---'] * len(frame.columns)) + '|']
    lines += ['| ' + ' | '.join(str(v) for v in row) + ' |' for row in frame.itertuples(index=False)]
    return '\n'.join(lines)


def pct(x, digits=0):
    return f'{x:.{digits}%}'


def prec_text(row, stratum):
    n = row[f'{stratum}_Y/(Y+N)']
    return f"{row[f'{stratum}_precision']:.0%} ({n})" if isinstance(n, str) and n else 'n/a'


# Term table: class, volume, precision with and without context
class_of = {**{t: 'stands alone' for t in STRONG_TERMS}, **{t: 'needs context' for t in CONTEXT_REQUIRED_TERMS}, **{t: 'dropped' for t in DROPPED_TERMS}}
term_memo = precision.assign(**{'class': precision['term'].map(class_of)})
term_memo = term_memo.assign(**{'with context': term_memo.apply(lambda r: prec_text(r, 'context'), axis=1),
                                'without context': term_memo.apply(lambda r: prec_text(r, 'no_context'), axis=1)})
term_memo = term_memo[['term', 'class', 'unique_emails', 'with context', 'without context']].rename(columns={'unique_emails': 'emails'})
term_memo['order'] = term_memo['class'].map({'stands alone': 0, 'needs context': 1, 'dropped': 2})
term_memo = term_memo.sort_values(['order', 'emails'], ascending=[True, False]).drop(columns='order')

gc = gate_cost.set_index('term')
lost_total = int(gate_cost['est_real_mentions_lost'].sum())
not_custodian = key_people.loc[key_people['status'] != 'custodian', 'person'].tolist()
is_custodian = key_people.loc[key_people['status'] == 'custodian', 'person'].tolist()
top_no_mailbox = int((nodes.head(TOP_NODES)['type'] == 'Enron address, no mailbox').sum())
seed_found = int(seed_table.groupby('seed')['found'].any().sum())
seed_in = int(seed_table['in_subset'].eq(True).groupby(seed_table['seed']).any().sum()) if 'in_subset' in seed_table else 0
top_people = ', '.join(f"{r.address.split('@')[0]} ({r.emails_handled})" for r in nodes.head(8).itertuples())
holders_top = ', '.join(f'{c} ({n})' for c, n in extra_counter.most_common(5))

memo_text = f"""# Enron EDA decision memo

Snapshot of cleaning run `{RUN_DIR.name}`, produced by `Enron_EDA.ipynb` (sections 1, 2, 3, 5 and 7). Tables and charts are in `eda_outputs/{RUN_DIR.name}/`.
Not done in this pass: monthly SPE rates with an event timeline (item 4) and contrastive term analysis (item 6).

## What the data looks like
- **{facts['unique_emails']:,} unique emails** ({facts['duplicate_copies']:,} further mailbox copies) from **{facts['custodians']} mailboxes** (after merging labels that differ only in case; {facts['custodian_labels']} labels in the raw data). Median mailbox: {facts['custodian_rows_median']:,} emails; the 10 largest hold {pct(facts['top10_custodian_share'])}.
- **Dates are unreliable at the edges.** Dates run from 1979 to 2044. {facts['outside_plot_range']:,} unique emails fall outside 1998 to 2002. Exclude or manually check them in any time-based step.
- **Most emails carry quoted text.** {pct(facts['share_with_quote_marker'], 1)} have a quote marker, and among those {pct(facts['share_quote_heavy'], 1)} are under half new text. {pct(facts['share_new_text_empty'], 1)} of emails have no new text at all (bare forwards).
- {pct(facts['share_external_sender'], 1)} of emails are from senders outside Enron (news, vendors) and {pct(facts['share_mass_mailing'], 1)} go to more than {MASS_MAILING_THRESHOLD} recipients.

## SPE term list: which names can be trusted
Precision is the share of sampled emails (10 per stratum) that use the name in the Enron SPE or structured-finance sense, from one reviewer working on short excerpts. Intervals are wide; see `term_precision.csv`.

{md_table(term_memo)}

## Decisions for the retrieval steps
| Decision | Evidence | Recommendation |
|---|---|---|
| **Which names stand alone** | Precision table above | Use the {len(STRONG_TERMS)} stand-alone names as direct filters. Require a financial-structure word (proposal Table 2 groups B and C) within about 100 words for {', '.join(CONTEXT_REQUIRED_TERMS)}. Drop {' and '.join(DROPPED_TERMS)} as stand-alone terms. |
| **Cost of those rules** | `term_class_cost.csv` | About {lost_total} real mentions are left out (rough estimate). Almost all of the gap is plain "Hawaii": an estimated {gc.at['hawaii', 'est_real_mentions_lost']:.0f} real mentions (range {gc.at['hawaii', 'approx_range']}) sit in emails with no "Hawaii 125" wording. Excerpts show the same structure under other names (McGarrett, Danno, Bali Trust); these are leads to source and test, not yet terms. |
| **Search text view** | {facts['spe_quoted_only']} of {facts['spe_unique']} SPE emails ({pct(facts['spe_quoted_only'] / facts['spe_unique'])}) mention a term only in quoted or forwarded text | Search the full text for recall, but rank and attribute using the new text, and show the whole thread to reviewers. A quoted-only match shows what the sender received, not what they wrote. |
| **How big the SPE pool is** | {facts['spe_unique']} unique emails ({pct(facts['spe_share_of_corpus'], 2)} of the corpus); {facts['spe_threads']} distinct subjects | The name-based pool is smaller than the proposal's 1,000-per-channel design. Treat it as one candidate source and keep the semantic and lexical channels for emails that never name a vehicle. |
| **Review pool** | Sender inside Enron, term in the sender's own text, not a mass mailing: {facts['spe_prime_total']} emails ({facts['spe_prime_before']} before the cutoff) | A workable size for human screening before any model-based reranking. |
| **Cutoff ({CUTOFF_LOCAL} Houston)** | {facts['spe_before_cutoff']} before, {facts['spe_after_cutoff']} on or after ({pct(facts['spe_after_share'], 1)} of the pool, against {pct(facts['corpus_after_share'], 1)} of the whole corpus); {facts['spe_near_cutoff']} within {NEAR_CUTOFF_HOURS} hours | The cutoff removes a large share of SPE discussion, including internal legal and finance email after the announcement. The group should decide whether to keep it for ranking and still read later internal emails as context. |
| **Duplicates** | {pct(facts['duplicate_copies'] / facts['rows'], 1)} of rows are copies; {facts['spe_with_copies']} SPE emails have a copy in another mailbox | Rank unique emails only; use `canonical_email_id` to list every mailbox that held a chosen email. |

## Who is in the data (section 5)
- **{len(not_custodian)} of the {len(key_people)} people named in the Powers report or the proposal examples have no mailbox here**: {', '.join(not_custodian)}. They appear only through other people's mail, mostly as recipients. {len(is_custodian)} do: {', '.join(is_custodian)}.
- For people without a mailbox, the corpus can show what they were sent, not what they wrote or did with it. Evidence of their awareness will come from replies and from emails written by others. The definition's tiers (all three elements in one thread, versus exposure and awareness with a presentation concern) should reflect this.
- **Who handles SPE emails most:** {top_people}. {top_no_mailbox} of the top {TOP_NODES} addresses have no mailbox in the data, so their view is partial and some are legal or finance staff reviewing deals. Handling many emails is not evidence of wrongdoing.
- **Mailbox holders who are not on the email:** {pct(facts['spe_unaddressed_share'], 1)} of SPE emails sit in a mailbox whose owner is neither sender nor addressee ({pct(facts['spe_unaddressed_strict_share'], 1)} under the stricter address map). Most frequent: {holders_top}. Possible causes include distribution lists, BCC, address aliases and forwarding; each case needs a look before it counts as evidence of receipt.
- **Seed emails:** {seed_found} of {len(SEEDS)} found, {seed_in} in the SPE pool. They shaped the term list, so this checks the pipeline, not its accuracy.

## Limits and open items
- Precision labels come from one reviewer, ten emails per stratum, about 200 characters of context. Ask the group to spot-check, especially for the context-required terms.
- Marlin, Trutta and Project Velocity are not in the Powers report (the proposal is their source). Timberwolf, Bobcat and Southampton are candidate additions; they need page citations before the paper uses them.
- Which addresses belong to which mailbox is inferred. Sent-folder evidence is strong; name matches are weaker. Address aliases are not merged, except two for Kaminski. Custodian labels with typos (for example `lavorado-j` next to `lavorato-j`) are still separate.
- Counts measure how often words appear, not wrongdoing. The network measures who handled emails, not who was responsible.
- Next steps from the original plan: monthly rates with an event timeline, and contrastive term analysis to find vocabulary that does not name a vehicle.
"""
memo_text += baseline_memo()
(PROJECT_ROOT / 'Enron_EDA_Decision_Memo.md').write_text(memo_text, encoding='utf-8')
(EDA_DIR / 'facts.json').write_text(json.dumps(facts, indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o)), encoding='utf-8')
print(f'Memo written: {PROJECT_ROOT / "Enron_EDA_Decision_Memo.md"} ({len(memo_text.split()):,} words)')

Memo written: D:\E_Guo\z_New_folder\INTA6450\enron_proj\Enron_EDA_Decision_Memo.md (1,501 words)
